In [1]:
from pathlib import Path
from stable_platform_matchings import InstanceGenerator, Optimizer, OptimizerParams, SolverOptions
from stable_platform_matchings.domain.instance import Instance
from stable_platform_matchings.graphs import RoadGraph
import numpy as np
import pickle

In [2]:
MIN_HET_COST = 200_000.0
MAX_HET_COST = 1_200_000.0

In [16]:
N_INTS = 28
SEED = 67

INDO_CRS = "EPSG:23867"
DATA_DIR = Path("../data")

FARMERS_PATH = DATA_DIR / "farmers.csv"
FARMERS_14_PATH = DATA_DIR / "farmers_14.csv"
INTS_PATH = DATA_DIR / "intermediaries.csv"
GRAPH_PATH = DATA_DIR / "graph_0-14960_00_new.pickle"
ALPHA_PATH = DATA_DIR / "precomputed_alpha.json"
SIGMAS_PATH = DATA_DIR / "precomputed_sigmas.json"

In [17]:
ig = InstanceGenerator(
    FARMERS_PATH, FARMERS_14_PATH, INTS_PATH, GRAPH_PATH, ALPHA_PATH, SIGMAS_PATH
)

In [5]:
ig.gen_intermediaries(n_intermediaries=N_INTS, seed=SEED)
ig.gen_calendar(seed=SEED, scale=1, n_cycles=10)

platform = ig.gen_instance(instance_id="hello", day=69, n_hist_sets=3)

In [3]:
platform = Instance.from_yaml("../data/anon_14_day_instances/aggregate_instance_1.yaml")
with open("../data/graph_0-14960_00_new.pickle", "rb") as f:
    graph = pickle.load(f)

platform.set_graph(RoadGraph(graph))

In [4]:
q = sum(f.quantity for f in platform.farmers)
q

45.6

In [5]:
tfv = q * platform.FRUIT_PRICE_PER_TON

tfv

114592800.0

In [6]:
epsilons = {'beautiful_bohr_2020-08-27': 2.0,
   'competent_mayer_2020-08-27': 2.0,
   'elated_haslett_2020-08-27': 2.0,
   'elegant_gagarin_2020-08-27': 2.0,
   'elegant_mendel_2020-08-27': 2.0,
   'exciting_fermi_2020-08-27': 2.0,
   'gallant_cerf_2020-08-27': 2.0,
   'hopeful_sanderson_2020-08-27': 2.0,
   'keen_visvesvaraya_2020-08-27': 2.0,
   'laughing_mestorf_2020-08-27': 2.0,
   'loving_engelbart_2020-08-27': 2.0,
   'peaceful_austin_2020-08-27': 2.0,
   'quizzical_elgamal_2020-08-27': 2.0,
   'vigorous_mccarthy_2020-08-27': 2.0}
het_costs = {'beautiful_bohr_2020-08-27': 208982.29188522615,
   'competent_mayer_2020-08-27': 460991.2051662722,
   'elated_haslett_2020-08-27': 305403.90175133076,
   'elegant_gagarin_2020-08-27': 437560.84311823314,
   'elegant_mendel_2020-08-27': 335559.2487460253,
   'exciting_fermi_2020-08-27': 635790.8779710636,
   'gallant_cerf_2020-08-27': 923373.3982204887,
   'hopeful_sanderson_2020-08-27': 840355.5704885144,
   'keen_visvesvaraya_2020-08-27': 1032911.7363867372,
   'laughing_mestorf_2020-08-27': 577101.8667446914,
   'loving_engelbart_2020-08-27': 518767.38177915645,
   'peaceful_austin_2020-08-27': 687357.3175807493,
   'quizzical_elgamal_2020-08-27': 966686.590492184,
   'vigorous_mccarthy_2020-08-27': 676889.1626719239}

In [7]:
params = OptimizerParams(
    het_costs=het_costs,
    epsilons=epsilons,
    backend="gurobi",
    vrp_mode="approximate",
    verbose=True,
    print_width=80,
    threads=14,
    vrp_time_limit_seconds=10,
)

opt = Optimizer(platform, params)



============================= Optimizer Parameters =============================
---------------------------------- het_costs -----------------------------------
  {'beautiful_bohr_2020-08-27': 208982.29188522615,
   'competent_mayer_2020-08-27': 460991.2051662722,
   'elated_haslett_2020-08-27': 305403.90175133076,
   'elegant_gagarin_2020-08-27': 437560.84311823314,
   'elegant_mendel_2020-08-27': 335559.2487460253,
   'exciting_fermi_2020-08-27': 635790.8779710636,
   'gallant_cerf_2020-08-27': 923373.3982204887,
   'hopeful_sanderson_2020-08-27': 840355.5704885144,
   'keen_visvesvaraya_2020-08-27': 1032911.7363867372,
   'laughing_mestorf_2020-08-27': 577101.8667446914,
   'loving_engelbart_2020-08-27': 518767.38177915645,
   'peaceful_austin_2020-08-27': 687357.3175807493,
   'quizzical_elgamal_2020-08-27': 966686.590492184,
   'vigorous_mccarthy_2020-08-27': 676889.1626719239}
----------------------------------- epsilons -----------------------------------
  {'beautiful_bohr_2

In [10]:
options = SolverOptions(
    strategy="heuristic_optimized",
    structured_farmer_payments=False,
    dominance_constraints=False,
    pay_unmatched=False,
    hist_set_method="instance_farmers",
    stabilize_branch_extrema=False,
    early_stop=True
)


summary = opt.solve(options=options)



================================ Solver Options ================================
  Seed                       0
  Strategy                   heuristic_optimized
  Structured Farmer Payments False
  Dominance Constraints      False
  Early Stop                 True
  Hist Set Method            instance_farmers
  Pay Unmatched              False
  Stabilize Branch Extrema   False


======================== Strategy: heuristic_optimized =========================
  Farmers                    25
  Intermediaries             14


============================== Branch Evaluation ===============================
  Forced matched:
    []
  Forced unmatched:
    []

----------------------------- Primal Solve Result ------------------------------
  Platform profit            1,108,806.334
  Max intermediary welfare   1,847,837.485
  Max farmer welfare         102,978,942.013
---------------------------- Lower-Bound Candidate -----------------------------
  Objective                  1,108,806.33

In [9]:
opt.active_hist_sets

{'hopeful_sanderson_2020-08-27': (frozenset(),),
 'competent_mayer_2020-08-27': (frozenset({'competent_mayer_14_1'}),),
 'beautiful_bohr_2020-08-27': (frozenset({'beautiful_bohr_14_12'}),),
 'elegant_mendel_2020-08-27': (frozenset({'elegant_mendel_23_8',
             'elegant_mendel_54_6',
             'elegant_mendel_69_3'}),),
 'elegant_gagarin_2020-08-27': (frozenset({'elegant_gagarin_16_1',
             'elegant_gagarin_98_1'}),),
 'quizzical_elgamal_2020-08-27': (frozenset({'quizzical_elgamal_58_2',
             'quizzical_elgamal_59_2',
             'quizzical_elgamal_59_3'}),),
 'vigorous_mccarthy_2020-08-27': (frozenset(),),
 'gallant_cerf_2020-08-27': (frozenset({'gallant_cerf_11_5',
             'gallant_cerf_12_12',
             'gallant_cerf_19_2',
             'gallant_cerf_29_4'}),),
 'elated_haslett_2020-08-27': (frozenset(),),
 'laughing_mestorf_2020-08-27': (frozenset({'laughing_mestorf_101_1',
             'laughing_mestorf_11_8',
             'laughing_mestorf_24_19'

In [27]:
2000000/14500

137.93103448275863